# CPA attack on standard ASCON software implementation

This notebook performs the **CPA attack** on the previously acquired power traces.  
It loads the traceset produced by the acquisition notebook ( the h5 file (.h5)) and computes:

- **Correlation Power Analysis (CPA)** results  
- **Key rank** vs. number of traces  
- **Correlation trends** for each key byte, comparing correct vs. wrong key hypotheses  

These plots help evaluate the **difficulty of recovering each key byte** and visualize the leakage behavior across the trace window.

⚠️ **Important:**  
This notebook assumes that the **Power Trace Acquisition** notebook (`xheep_capture_ASCON.ipynb`) has already been executed and the traceset has been acquired.


In [1]:
sbox_type   = "lut_ascon"   # do not modify this line
tested_sbox = sbox_type     # alias used later in the printout
n_trc       = 10_000        # total number of traces in the traceset

In [2]:
import sys
import os
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import logging
import json
import h5py

## Project paths
This block robustly detects the project root (`DOJO_ROOT`) starting from either the script location (`__file__`) or the current working directory (for notebooks). From there it defines all relevant subdirectories (ASCON sources, SCA scripts, X-HEEP, traces, plots, cache), ensures output folders exist, and adds the local source paths to `sys.path` .

In [3]:
def find_project_root(start: Path, markers=("fusesoc.conf", ".dojo_root")) -> Path:
    current = start
    while current != current.parent:
        if any((current / m).exists() for m in markers):
            return current
        current = current.parent

    raise RuntimeError(
        f"Could not find project root (looked for markers: {markers}). "
        "Please ensure you are inside the Side-Channel-Dojo repository."
    )

# In a script, __file__ exists; in a notebook it does not.
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

DOJO_ROOT = find_project_root(SCRIPT_DIR)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

ASCON_PY_DIR = DOJO_ROOT / "sw" / "ciphers" / "ASCON_init_python"
SCA_DIR      = DOJO_ROOT / "sw" / "sca_scripts"
HW_DIR       = DOJO_ROOT / "hw"

# Base dirs for ASCON SW SCA
BASE_PLOT_DIR  = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "plot"
BASE_CACHE_DIR = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "cache"

# Traceset (HDF5) for ASCON SW
TRACESET_DIR  = DOJO_ROOT / "sw" / "traceset" / "ASCON" / "sw"
TRACESET_FILE = TRACESET_DIR / f"ascon_opt32_{sbox_type}_{n_trc // 1000}k.h5"

# Per-S-box plot and cache dirs
PLOT_DIR       = BASE_PLOT_DIR / sbox_type
CPA_CACHE_FILE = BASE_CACHE_DIR / sbox_type / f"CPA_results_{sbox_type}.json"

# Ensure directories exist
BASE_PLOT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
CPA_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

# Import local modules
sys.path.insert(0, str(ASCON_PY_DIR))
sys.path.insert(0, str(SCA_DIR))

## Imports

In [4]:
from analyzer.attack.ascon.xheep_ascon_cpa.ascon_generic_leakage_model import ascon_generic_leakage_model
from analyzer.attack.ascon.xheep_ascon_cpa.ascon_cpa import ascon_cpa

# Configuration

In [5]:
# ---------------------------------------------------------------------------
# Flow flags
# ---------------------------------------------------------------------------
verbose                  = True   # Verbose output during key recovery

# CPA / analysis cache control
load_attack_results      = False  # Load CPA cache if available
save_attack_results      = True   # Save CPA results to cache after the run

# Plot control
key_rank_plot            = True   # Plot PGE vs traces
traces_correlation_plot  = True   # Plot correlation vs traces

# Output control
save_plots               = True   # Save plots to disk
save_results             = True   # Save analysis results (JSON, etc.) to disk

# ---------------------------------------------------------------------------
# Configuration printout
# ---------------------------------------------------------------------------

def _yn(flag: bool) -> str:
    """Return 'yes' or 'no' for a boolean flag."""
    return "yes" if flag else "no"

print("\n================= CONFIGURATION =================")
print(f"DOJO_ROOT           : {DOJO_ROOT}")
print()
print("Target")
print(f"  Cipher                    : ASCON ")
print(f"  S-box implementation      : {tested_sbox}")
print(f"  Notebook scope            : Attacked ASCON SW with generic S-box")
print()
print("Paths")
print(f"  Traceset file             : {TRACESET_FILE}")
print(f"  Plot dir                  : {PLOT_DIR}")
print(f"  CPA cache file            : {CPA_CACHE_FILE}")
print()
print("Analysis configuration")
print(f"  Total traces (n_trc)      : {n_trc}")
print(f"  Save plots                : {_yn(save_plots)}")
print(f"  Save results              : {_yn(save_results)}")
print(f"  Load CPA results (cache)  : {_yn(load_attack_results)}")
print(f"  Save CPA results (cache)  : {_yn(save_attack_results)}")
print()
print(f"  Key rank plot             : {_yn(key_rank_plot)}")
print(f"  Correlation plot          : {_yn(traces_correlation_plot)}")
print("=================================================\n")


================= CONFIGURATION =================
DOJO_ROOT           : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo

Target
  Cipher                    : ASCON 
  S-box implementation      : lut_ascon
  Notebook scope            : Attacked ASCON SW with generic S-box

Paths
  Traceset file             : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/traceset/ASCON/sw/ascon_opt32_lut_ascon_10k.h5
  Plot dir                  : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/sca_scripts/ASCON/sw/plot/lut_ascon
  CPA cache file            : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/sca_scripts/ASCON/sw/cache/lut_ascon/CPA_results_lut_ascon.json

Analysis configuration
  Total traces (n_trc)      : 10000
  Save plots                : yes
  Save results              : yes
  Load CPA results (cache)  : no
  Save CPA results (cache)  : yes

  Key rank plot             : yes
  Correlation plot          : yes



## SCA attack

## Load data to perform the attack
- Opens the HDF5 traces file `TRACESET_FILE` in read mode.
- Checks that both datasets `"traces"` and `"nonces"` are present.
- Reads:
  - Up to `n_trc` traces and their corresponding nonces.
  - Metadata: `sampling_interval`, `n_samples`, `key`, `iv`.
- Verifies that the number of traces matches the number of nonces.
- Warns if the requested `n_trc` differs from the number of traces stored.


In [6]:
try:
    with h5py.File(TRACESET_FILE, "r") as f_read_traces:
        # Basic sanity: check that required datasets exist
        if "traces" not in f_read_traces or "nonces" not in f_read_traces:
            raise KeyError(
                "HDF5 file is missing required datasets 'traces' and/or 'nonces'."
            )

        traces_ds = f_read_traces["traces"]
        nonces_ds = f_read_traces["nonces"]

        total_traces = traces_ds.shape[0]
        n_samples    = traces_ds.shape[1]

        # Load metadata attributes (if present)
        sampling_interval   = f_read_traces.attrs.get("sampling_interval", None)
        n_samples           = f_read_traces.attrs.get("n_samples", n_samples)
        key                 = f_read_traces.attrs.get("key_hex", None) 
        iv                  = f_read_traces.attrs.get("iv_hex", None)

        # Use at most n_trc traces, but do not exceed what's in the file
        n_used = min(n_trc, total_traces)

        # Use slicing so data are actually loaded into RAM
        traces = traces_ds[:n_used]
        nonces = nonces_ds[:n_used]

    # Sanity check: traces and nonces should have the same number of rows
    if traces.shape[0] != nonces.shape[0]:
        raise ValueError(
            f"Number of traces ({traces.shape[0]}) and nonces ({nonces.shape[0]}) "
            "do not match. Check the traces file."
        )

    # Optional sanity checks vs metadata
    if n_trc != total_traces:
        print(
            f"[WARN] Wanted n_trc={n_trc} "
            f"differs from dataset length={total_traces}"
        )
        
        
    nonce_msb = int(nonces[0, 0])  # 0x0F0E0D0C0B0A0908
    nonce_lsb = int(nonces[0, 1])  # 0x0706050403020100
    # Rebuild the 128-bit value as stored (byte-reversed version)
    nonce_stored_int = (nonce_msb << 64) | nonce_lsb
    # Undo the byte reversal to recover the original big-endian nonce
    nonce_original_bytes = nonce_stored_int.to_bytes(16, byteorder="big")[::-1]
    nonce_original_hex = nonce_original_bytes.hex().upper()

    nonce_ini_hex = nonces[0]
    print(f"[INFO] Loaded {traces.shape[0]} traces from {TRACESET_FILE}")
    print(f"[INFO] Sampling interval      : {sampling_interval}")
    print(f"[INFO] Samples per trace      : {n_samples}")
    print(f"[INFO] Key                    : 0x{key}")
    print(f"[INFO] IV                     : 0x{iv}")
    print(f"[INFO] Initial nonce          : 0x{nonce_original_hex}")

except FileNotFoundError:
    print(
        f"[ERROR] Traces file {TRACESET_FILE} not found. "
        "Please run the trace acquisition phase first."
    )
except Exception as e:
    print(f"[ERROR] Could not read traces file {TRACESET_FILE}: {e}")


[INFO] Loaded 10000 traces from /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/traceset/ASCON/sw/ascon_opt32_lut_ascon_10k.h5
[INFO] Sampling interval      : 8e-09
[INFO] Samples per trace      : 1125
[INFO] Key                    : 0x000102030405060708090A0B0C0D0E0F
[INFO] IV                     : 0x00001000808C0001
[INFO] Initial nonce          : 0x000102030405060708090A0B0C0D0E0F


### Single-bit CPA attack on standard ASCON S-box

In the standard ASCON S-box, each output bit can be written in Algebraic Normal Form (ANF), i.e., as a Boolean polynomial over $GF(2)$.  
For power/EM analysis we are interested **only in nonlinear key–nonce products**, because they modulate the switching activity in a key-dependent way.

From the ANF, and grouping $x_1$ and $x_2$ as an indistinguishable term $x_{12} = x_1 \oplus x_2$, we rename:
- $x_1 \to k''$ (first half of the key)
- $x_{12} \to k'$ (combined key term)
- $x_3 \to m''$, $x_4 \to m'$ (nonce halves)

The useful S-box outputs are the ones containing nonlinear mixes like $m'' \cdot k''$ or $m'' \cdot k'$; purely nonce-only terms (e.g. $m' \cdot (m'' + 1)$) are **not** exploitable for key recovery.

A crucial observation from the ANF is that, for the chosen attacked bit of $x_0$ after the first round, the **MSB half of the key does not appear** in the nonlinear term.  
This enables a **two-phase attack**:

1. **Phase 1 – Attack $x_0$ (recover 3 bits of the first key half)**  
   - Target a bit whose expression contains a term like $m'' \cdot k''$.  
   - Only 3 key bits influence this bit in a nonlinear way, so we treat **exactly 3 bits of $k''$ as unknown** and keep the rest as constants.  
   - By CPA on that bit, we recover those 3 key bits.

2. **Phase 2 – Attack $x_1$ (recover 3 bits of the second key half)**  
   - The 3 bits recovered from $x_0$ are now treated as known.  
   - We move to a bit in $x_1$ whose ANF contains a different triple of key bits (second half).  
   - Again, only 3 bits are unknown, leading to another 8 hypotheses and a second CPA step to recover them.

---

### Leakage model construction (standard S-box, 3-bit model)

For a **single attacked bit** after the first-round substitution + diffusion:

- At any time, we assume **only 3 key bits are unknown**.
- This gives **8 hypotheses**: all 3-bit values $k \in \{0, \dots, 7\}$.

For each trace (indexed by $n$):

1. Extract the nonce halves:
   - `nonce_MSB = nonces[n][1]`  
   - `nonce_LSB = nonces[n][0]`
2. Call  
   `leakage_model = ascon_leakage_model(init_vect, nonce_MSB, nonce_LSB, state_register_index, attacked_bit, sbox_type, key_0=...)`
3. This returns an array of length 8:
   - `leakage_model[k]` is the predicted value (0/1) of the attacked bit for key guess $k$.

Stacking this over $N$ traces yields the hypothetical leakage matrix:
- `H_matrix` of shape $(N, 8)$:
  - rows: traces / nonces
  - columns: 3-bit key guesses $k = 0..7$

This is the **standard-ASCON leakage model**: a single-bit selection function mapping each nonce and 3-bit key guess to a predicted output bit.

---

### CPA step on a single bit

Given:
- `traces` of shape $(N, M)$ (N traces, M samples per trace)
- `H_matrix` of shape $(N, 8)$

the function `ascon_cpa(traces, H_matrix)`:

1. For each key guess $k$ and each time sample $t$:
   - `x = traces[:, t]` → measured power at time $t$  
   - `y = H_matrix[:, k]` → predicted leakage for guess $k$  
   - compute Pearson correlation:
     - $R[k, t] = \mathrm{corr}(x, y)$

2. This yields `R_matrix` of shape $(8, M)$.

3. To score each key guess:
   - `corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1)`  
   - one value per 3-bit hypothesis (max absolute correlation across time samples)

As the number of traces grows, the **correct 3-bit key guess** should exhibit the **largest correlation**, allowing us to recover 3 bits of the key from a single attacked state bit in the first round.


In [7]:
# Reverse key by bytes (2 hex chars per byte) to match C endianness
key_bytes       = [key[i:i+2] for i in range(0, len(key), 2)]
key_reversed    = "".join(key_bytes[::-1])
key_int         = int(key_reversed, 16)

# IV as integer (already in correct endianness)
iv_int = int(iv, 16)

In [ ]:
k0_int  = (key_int >> 64) & 0xFFFFFFFFFFFFFFFF  # 0001020304050607
k1_int  =  key_int        & 0xFFFFFFFFFFFFFFFF  # 08090A0B0C0D0E0F

# Expected bits, LSB at index 0
k0_bits = np.array([(k0_int >> i) & 1 for i in range(64)], dtype=np.uint8)
k1_bits = np.array([(k1_int >> i) & 1 for i in range(64)], dtype=np.uint8)
print(f"[INFO] k0 expected        : {}")
print(f"[INFO] k1 expected        : {}")

[INFO] k0 expected        : 0xf0e0d0c0b0a0908
[INFO] k1 expected        : 0x706050403020100


# Recovering the full key
TO recover the full key first the key index are taken by running the SNR to select which among the register x0 or x1 bits has the highest value and can lead to better attack

In [10]:
verbose = False
attacked_targets = [("x0", 32), ("x1", 32), ("x0", 13), ("x1", 0), ("x0", 34), ("x1", 63)]

In [ ]:
full_key_recovered = False

# -------------------------------------------------------------------
# Containers for recovered key bits
# -------------------------------------------------------------------
k0_bits      = np.zeros(64, dtype=np.uint8)   # most significant 64 bits
k0_recovered = np.zeros(64, dtype=bool)

k1_bits      = np.zeros(64, dtype=np.uint8)   # least significant 64 bits
k1_recovered = np.zeros(64, dtype=bool)

tic = time.perf_counter()
print("\n=== Recovering k0 and k1 (generic 6-bit leakage model) ===\n")

# attacked_targets: list of (state_name, attacked_bit)
# Example: [("x0", 32), ("x0", 13), ("x1", 19), ("x2", 28), ...]

for attacked_state_reg, attacked_bit in tqdm(attacked_targets, desc="Key bit recovery"):
    if full_key_recovered:
        break

    # ---------------------------------------------------------------
    # 1) Determine which 3 global bit positions this attack constrains
    #    (same indices used for k0 and k1)
    # ---------------------------------------------------------------
    j = attacked_bit % 64
    if attacked_state_reg == "x0":
        k_idx = np.array([j, (j + 19) % 64, (j + 28) % 64], dtype=int)
    elif attacked_state_reg == "x1":
        k_idx = np.array([j, (j + 61) % 64, (j + 39) % 64], dtype=int)
    elif attacked_state_reg == "x2":
        k_idx = np.array([j, (j + 1) % 64, (j + 6) % 64], dtype=int)
    elif attacked_state_reg == "x3":
        k_idx = np.array([j, (j + 10) % 64, (j + 17) % 64], dtype=int)
    elif attacked_state_reg == "x4":
        k_idx = np.array([j, (j + 7) % 64, (j + 41) % 64], dtype=int)
    else:
        raise ValueError(f"Invalid attacked_state_register: {attacked_state_reg}")

    # ---------------------------------------------------------------
    # 2) Build full leakage matrix H for this (state, bit)
    #    H has shape (n_traces, 64):
    #      - rows    = traces / nonces
    #      - columns = 6-bit key hypotheses (3 bits in k0, 3 bits in k1)
    # ---------------------------------------------------------------
    n_traces = nonces.shape[0]
    H = np.empty((n_traces, 64), dtype=np.uint8)

    for n in range(n_traces):
        nonce_lsb = int(nonces[n, 0])  # LSB 64 bits
        nonce_msb = int(nonces[n, 1])  # MSB 64 bits

        H[n, :] = ascon_generic_leakage_model(
            iv_int,
            nonce_msb,
            nonce_lsb,
            attacked_state_reg,
            attacked_bit,
            sbox_type,
        )

    # ---------------------------------------------------------------
    # 3) Group hypotheses that produce the same leakage pattern:
    #
    #    - unique_cols: shape (num_groups, n_traces)
    #        Each row is a distinct leakage pattern across all traces.
    #        If hypotheses 3 and 17 produce the same leakage vector,
    #        these key hypothesis cannot be distinghuished in the attack. 
    #
    #    - group_map: length 64
    #        group_map[k] = g  means "hypothesis k belongs to group g",
    #        and that group’s leakage pattern is unique_cols[g, :].
    # ---------------------------------------------------------------
    unique_cols, group_map = np.unique(H.T, axis=0, return_inverse=True)
    H_group = unique_cols.T   # shape (n_traces, num_groups)

    num_groups = H_group.shape[1]
    key_hyp_groups = [[] for _ in range(num_groups)]

    for k_guess, g_idx in enumerate(group_map):
        key_hyp_groups[g_idx].append(k_guess)

    # Optional: pretty-print all groups
    if verbose:
        print(f"\nTotal number of groups > {num_groups}\n")
        print("group   hyp   k0    k1")
        print("------------------------")
        for g, members in enumerate(key_hyp_groups):
            if not members:
                continue
            for line_idx, hyp_idx in enumerate(members):
                # 6-bit hypothesis index: [k0_2 k0_1 k0_0 k1_2 k1_1 k1_0]
                k0_loc = (hyp_idx >> 3) & 0b111  # upper 3 bits
                k1_loc = hyp_idx        & 0b111  # lower 3 bits
                group_label = f"{g:5d}" if line_idx == 0 else " " * 5
                print(f"{group_label}  {hyp_idx:3d}  {k0_loc:03b}  {k1_loc:03b}")
            print()

    # ---------------------------------------------------------------
    # 4) Run CPA once per *grouped* hypothesis
    #    ascon_cpa expects:
    #      - traces: shape (N, M)
    #      - hypothetical_values: shape (N, K)
    #    Here K = num_groups.
    # ---------------------------------------------------------------
    R_group   = ascon_cpa(traces, H_group)         # shape (num_groups, n_samples)
    corr_group = np.max(np.abs(R_group), axis=1)   # got |corr| per each group

    best_group_idx = int(np.argmax(corr_group))
    best_key_group   = key_hyp_groups[best_group_idx]   

    # ---------------------------------------------------------------
    # 5) Decode local 6 bits (3 for k0, 3 for k1) for each hypothesis
    #    in the best group, and see which bits are *constant* across
    #    the group (those are the bits we can actually recover).
    #
    #    Encoding convention for hypothesis index h (0..63):
    #        h = (b5 b4 b3 b2 b1 b0)_2
    #        - local k0 bits = (b5, b4, b3)
    #        - local k1 bits = (b2, b1, b0)
    #
    #    We want local indices i=0,1,2 to map as:
    #        i=0 -> bit for global index k_idx[0] (j)
    #        i=1 -> bit for global index k_idx[1] (j+Δ1)
    #        i=2 -> bit for global index k_idx[2] (j+Δ2)
    # ---------------------------------------------------------------
    k0_local_list = []
    k1_local_list = []

    for key_hyp in best_key_group:
        # local k0 bits: [b5, b4, b3]
        k0_local = np.array(
            [(hyp_idx >> (5 - b)) & 1 for b in range(3)], dtype=np.uint8
        )
        # local k1 bits: [b2, b1, b0]
        k1_local = np.array(
            [(hyp_idx >> (2 - b)) & 1 for b in range(3)], dtype=np.uint8
        )

        k0_local_list.append(k0_local)
        k1_local_list.append(k1_local)

    k0_local_all = np.vstack(k0_local_list)  # shape (num_members, 3)
    k1_local_all = np.vstack(k1_local_list)  # shape (num_members, 3)

    # ---------------------------------------------------------------
    # 6) For each of the 3 recovered bit positions:
    #    - Look at the best group of hypotheses (best_members).
    #    - For that group:
    #         k0_local_all: shape (num_members, 3)
    #         k1_local_all: shape (num_members, 3)
    #      where column 'i' is the local bit value for all members.
    #    - If all hypotheses in the best group agree on local bit i
    #      for k0 or k1, that bit is distinguishable → we can store it
    #      into global position k_idx[i], unless we already recovered
    #      a conflicting value earlier.
    # ---------------------------------------------------------------
    for i in range(3):
         # key index in the 64-bit word (same index used for k0 and k1)
        key_idx = int(k_idx[i])  

        # ---------- k0 half ----------
         # check if the bit recovered for k0 at local position i is constant
        # across all hypotheses in the best group, meaning we can recover it
        bits_k0_group = k0_local_all[:, i] 

        if np.all(bits_k0_group == bits_k0_group[0]):
            val0 = int(bits_k0_group[0])

            if not k0_recovered[key_idx]:
                k0_bits[key_idx] = val0
                k0_recovered[key_idx] = True
                print(f"[INFO] Recovered k0 bit {key_idx}: {val0}, expected: {k0_bits[key_idx]}")
            elif k0_bits[key_idx] != val0 and verbose:
                print(
                    f"[WARN] Conflicting recovery for k0 bit {key_idx}: "
                    f"existing={k0_bits[key_idx]}, new={val0} (ignored)"
                )

        # ---------- k1 half ----------
        # check if the bit recovered for k1 at local position i is constant
        # across all hypotheses in the best group, meaning we can recover it
        bits_k1_group = k1_local_all[:, i]

        if np.all(bits_k1_group == bits_k1_group[0]):
            val1 = int(bits_k1_group[0])

            if not k1_recovered[key_idx]:
                k1_bits[key_idx] = val1
                k1_recovered[key_idx] = True
                print(f"[INFO] Recovered k1 bit {key_idx}: {val1}, expected: {k1_bits[key_idx]}")
            elif k1_bits[key_idx] != val1 and verbose:
                print(
                    f"[WARN] Conflicting recovery for k1 bit {key_idx}: "
                    f"existing={k1_bits[key_idx]}, new={val1} (ignored)"
                )
    print("Number of recovered bits so far: "
          f"k0 = {np.sum(k0_recovered)}/64, "
          f"k1 = {np.sum(k1_recovered)}/64")

    # -----------------------------------------------------------
    # After updating the three local bits: check if key is complete
    # -----------------------------------------------------------
    if np.all(k0_recovered) and np.all(k1_recovered):
        full_key_recovered = True
        if verbose:
            print("[INFO] All key bits recovered – stopping early.")

toc = time.perf_counter()
print(f"\nFull key recovery attack completed in {(toc - tic)/60:.2f} minutes.")


000102030405060708090A0B0C0D0E0F



Recovered most significant half of the key (k0): 4000000000000000

Recovered least significant half of the key (k1): 0000000000000000

Recovered full key (k1||k0): 00000000000000004000000000000000
Recovered key (little-endian): 0x00000000000000400000000000000000
ERROR: Key recovery faiXled.
  Got     : 0x00000000000000400000000000000000
  Expected: 0x000102030405060708090A0B0C0D0E0F

Full key recovery phase completed in 1.11 minutes.
